In [4]:
!{sys.executable} -m pip install openmeteo-requests requests-cache retry-requests
import sys
import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
	"latitude": 8.4855,
	"longitude": 76.9492,
	"start_date": "2000-01-01",
	"end_date": "2025-12-31",
	"hourly": "temperature_2m",
	"timezone": "GMT",

}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {
	"date": pd.date_range(
		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
		freq = pd.Timedelta(seconds = hourly.Interval()),
		inclusive = "left"
	)
}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)

Defaulting to user installation because normal site-packages is not writeable
Coordinates: 8.471001625061035°N 76.9329833984375°E
Elevation: 12.0 m asl
Timezone: NoneNone
Timezone difference to GMT+0: 0s

Hourly data
                             date  temperature_2m
0      2000-01-01 00:00:00+00:00       22.967001
1      2000-01-01 01:00:00+00:00       22.967001
2      2000-01-01 02:00:00+00:00       23.617001
3      2000-01-01 03:00:00+00:00       24.567001
4      2000-01-01 04:00:00+00:00       25.467001
...                          ...             ...
227923 2025-12-31 19:00:00+00:00       25.150000
227924 2025-12-31 20:00:00+00:00       24.950001
227925 2025-12-31 21:00:00+00:00       24.600000
227926 2025-12-31 22:00:00+00:00       24.250000
227927 2025-12-31 23:00:00+00:00       24.250000

[227928 rows x 2 columns]


In [3]:
import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry

# Setup Open-Meteo API client
cache_session = requests_cache.CachedSession(
    ".cache",
    expire_after=-1
)

retry_session = retry(
    cache_session,
    retries=5,
    backoff_factor=0.2
)

openmeteo = openmeteo_requests.Client(
    session=retry_session
)

# Open-Meteo Historical Weather API
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 8.4855,
    "longitude": 76.9492,
    "start_date": "2000-01-01",
    "end_date": "2025-12-31",

    "hourly": (
        "temperature_2m,"
        "relative_humidity_2m,"
        "surface_pressure,"
        "wind_speed_10m,"
        "wind_direction_10m,"
        "precipitation,"
        "cloud_cover,"
        "weather_code"
    ),

    "timezone": "GMT"
}

responses = openmeteo.weather_api(url, params=params)

response = responses[0]

print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}")
print(f"UTC offset: {response.UtcOffsetSeconds()}s")

# Hourly data
hourly = response.Hourly()

hourly_data = {
    "date": pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left"
    ),

    "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
    "relative_humidity_2m": hourly.Variables(1).ValuesAsNumpy(),
    "surface_pressure": hourly.Variables(2).ValuesAsNumpy(),
    "wind_speed_10m": hourly.Variables(3).ValuesAsNumpy(),
    "wind_direction_10m": hourly.Variables(4).ValuesAsNumpy(),
    "precipitation": hourly.Variables(5).ValuesAsNumpy(),
    "cloud_cover": hourly.Variables(6).ValuesAsNumpy(),
    "weather_code": hourly.Variables(7).ValuesAsNumpy()
}

weather_df = pd.DataFrame(hourly_data)

print("\nDataset shape:")
print(weather_df.shape)

print("\nColumns:")
print(weather_df.columns.tolist())

print("\nWeather code distribution:")
print(weather_df["weather_code"].value_counts().sort_index())

print("\nMissing values:")
print(weather_df.isnull().sum())

# Save as a NEW file — don't overwrite your original
weather_df.to_csv("weather_data_with_code.csv", index=False)

print("\nSaved as: weather_data_with_code.csv")

Coordinates: 8.471001625061035°N 76.9329833984375°E
Elevation: 12.0 m asl
Timezone: None
UTC offset: 0s

Dataset shape:
(227928, 9)

Columns:
['date', 'temperature_2m', 'relative_humidity_2m', 'surface_pressure', 'wind_speed_10m', 'wind_direction_10m', 'precipitation', 'cloud_cover', 'weather_code']

Weather code distribution:
weather_code
0.0     27499
1.0     26447
2.0     22101
3.0     66934
51.0    54032
53.0    15557
55.0     4005
61.0     6725
63.0     4287
65.0      341
Name: count, dtype: int64

Missing values:
date                    0
temperature_2m          0
relative_humidity_2m    0
surface_pressure        0
wind_speed_10m          0
wind_direction_10m      0
precipitation           0
cloud_cover             0
weather_code            0
dtype: int64

Saved as: weather_data_with_code.csv
